# California County Climate Report

Generates a one-page, report-style climate summary for any of California's
58 counties using `climakitae.ClimateData().get()` and the reusable
rendering components in `climakitae.visualize`.

**What you get**

| Panel | Description |
|---|---|
| Stat cards | Three headline numbers (avg-high rise, heat-wave frequency, extra hot days) |
| Summary table | Metric × period table: Historic · Near · Mid · Late century |
| Bar chart | Countywide 1-in-10-yr extreme heat threshold: historical vs late-century |

**Data**: WRF dynamical downscaling — daily `t2max` / `t2min`, 9 km grid
(`d02`), converted to °F, clipped to the selected county.

**Periods (all global warming levels, ±15 yr window per ensemble member)**

| Period | GWL |
|---|---|
| Historic baseline | 0.8 °C |
| Near-century      | 1.5 °C |
| Mid-century       | 2.0 °C |
| Late-century      | 3.0 °C |

> **Caveat** — The 1-in-10-yr threshold is an annual-max-quantile estimate.
> For rigorous return-period analysis, use the `metric_calc one_in_x`
> processor (GEV fit), preferably on LOCA2 for better tail estimates.
> WRF ensemble size is smaller than LOCA2; treat summaries as central
> tendencies, not the full uncertainty range.

## 0 · Install `climakitae.visualize`

Until [cal-adapt/climakitae#765](https://github.com/cal-adapt/climakitae/pull/765)
is merged, install directly from the feature branch:

In [ ]:
# binder/postBuild
!pip install git+https://github.com/cal-adapt/climakitae.git@feature/report-visualize --no-deps

## 1 · Imports

In [ ]:
from __future__ import annotations

import warnings

import ipywidgets as widgets
import pandas as pd
from IPython.display import Image, display
import xarray as xr

from climakitae.new_core.user_interface import ClimateData
from climakitae.visualize import (
    PeriodInputs,
    build_report_figure,
    compute_report_metrics,
)

warnings.filterwarnings("ignore", category=FutureWarning)


## 2 · Choose county and thresholds

In [ ]:
from climakitae.new_core.data_access import DataCatalog

CA_COUNTIES = sorted(DataCatalog().boundaries._ca_counties["NAME"].tolist())

county_w = widgets.Dropdown(
    options=CA_COUNTIES,
    value="Sacramento County",
    description="County:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "initial"},
)
hot_day_w = widgets.IntSlider(
    value=90, min=80, max=115, step=1,
    description="Hot-day threshold (\u00b0F):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
heatwave_w = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description="Heat-wave min consecutive days:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

display(widgets.VBox([
    widgets.HTML("<b>Select county and thresholds, then run the cells below.</b>"),
    county_w,
    hot_day_w,
    heatwave_w,
]))

WARMING_LEVELS = [0.8, 1.5, 2.0, 3.0]
PERIOD_LABELS = {
    0.8: "Historic\n(GWL 0.8\u00b0C)",
    1.5: "Near-future\n(GWL 1.5\u00b0C)",
    2.0: "Mid-century\n(GWL 2.0\u00b0C)",
    3.0: "Late-century\n(GWL 3.0\u00b0C)",
}

## 3 · Resolve selections

Snapshot widget values into plain Python so the rest of the notebook is
deterministic when re-run end-to-end.

In [ ]:
# Set to False to recompute from raw climate data instead of serving the pre-rendered PNG.
USE_PRERENDERED = True

county: str = county_w.value
hot_day_threshold_F: int = int(hot_day_w.value)
heatwave_min_days: int = int(heatwave_w.value)
warming_levels: list[float] = WARMING_LEVELS

print(f"County                    : {county}")
print(f"Hot-day threshold         : >{hot_day_threshold_F} °F")
print(f"Heat-wave minimum days    : {heatwave_min_days}")
print(f"Warming levels            : {warming_levels} °C")
print(f"Use pre-rendered PNG      : {USE_PRERENDERED}")


In [ ]:
_S3_REPORT_BASE = "https://cadcat.s3.amazonaws.com/county-reports/extreme-heat"


def _county_slug(county: str) -> str:
    """Convert a county name to a URL-safe slug, e.g. 'Los Angeles County' → 'los_angeles'."""
    return county.lower().replace(" county", "").strip().replace(" ", "_")


def display_county_report(county: str) -> None:
    """Display the pre-rendered county extreme-heat climate report from S3.

    Falls back with an informative message if the image is not yet available.
    """
    url = f"{_S3_REPORT_BASE}/{_county_slug(county)}.png"
    display(Image(url=url, width=1100))


In [ ]:

# When USE_PRERENDERED is True, show the S3 image and stop the execution queue.
# Jupyter stops cleanly — the kernel stays alive, no cells below will run.
# Set USE_PRERENDERED = False in the resolve cell above to recompute from raw data.
if USE_PRERENDERED:
    import base64
    import urllib.request
    from IPython.display import HTML

    display_county_report(county)

    # Fetch the image server-side and trigger a browser download via a data URL.
    # Using a data URL sidesteps cross-origin restrictions that block the anchor
    # `download` attribute on third-party URLs (e.g. S3).
    _img_url = f"{_S3_REPORT_BASE}/{_county_slug(county)}.png"
    _filename = f"cal-adapt_{_county_slug(county)}_extreme_heat_report.png"
    with urllib.request.urlopen(_img_url) as _resp:
        _b64 = base64.b64encode(_resp.read()).decode()
    display(HTML(f"""<script>
(function () {{
    var a = document.createElement('a');
    a.href = 'data:image/png;base64,{_b64}';
    a.download = '{_filename}';
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
}})();
</script>"""))

    raise KeyboardInterrupt


## 4 · Fetch all warming-level windows

WRF daily `t2max` and `t2min`, 9 km grid, clipped to the selected county,
converted to °F. The historical baseline is GWL 0.8 °C; near/mid/late
century are 1.5, 2.0, and 3.0 °C respectively. climakitae selects a
±15-yr window around each ensemble member's crossing of each level.
*(Expect ~30 s – 3 min per level depending on data-store latency.)*

In [ ]:
def _fetch_gwl(variable: str, levels: list[float]) -> xr.Dataset:
    """Fetch one WRF variable for all warming level windows.

    Returns an xr.Dataset with dimensions (sim, warming_level, time, lat, lon).
    ``add_dummy_time=True`` replaces the ``time_delta`` offset axis with a
    synthetic daily ``time`` coordinate so the metric helpers can use
    ``groupby("time.year")`` and ``dt.month`` filtering.
    """
    return (
        ClimateData(verbosity=-1)
        .catalog("cadcat")
        .variable(variable)
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .processes({
            "warming_level": {"warming_levels": levels, "add_dummy_time": True},
            "clip": county,
            "convert_units": "degF",
        })
        .get()
    )

print("Fetching GWLs …")
gwl_data: dict[str, xr.Dataset] = {
    "tmax": _fetch_gwl("t2max", list(warming_levels)),
    "tmin": _fetch_gwl("t2min", list(warming_levels)),
}
print("Done. Variables fetched:", list(gwl_data))


In [ ]:
# Filter out EC-Earth3-Veg simulations
def filter_sims(ds, exclude="ec-earth3-veg"):
    """Drop simulations containing the exclude string (case-insensitive)."""
    sim_strs = [str(s) for s in ds.sim.values]
    keep_mask = [exclude not in s.lower() for s in sim_strs]
    return ds.isel(sim=keep_mask)

for var, df in gwl_data.items():
    gwl_data[var] = filter_sims(df)


## 5 · Compute report metrics

`compute_report_metrics` accepts an ordered dict of period label →
`PeriodInputs(tmax, tmin)` and returns a tidy `DataFrame` (metrics as rows,
periods as columns) that the renderers consume directly.

In [ ]:
tmax_var = list(gwl_data["tmax"].data_vars)[0]
tmin_var = list(gwl_data["tmin"].data_vars)[0]

periods: dict[str, PeriodInputs] = {}
for level in warming_levels:
    periods[PERIOD_LABELS[level]] = PeriodInputs(
        tmax=gwl_data["tmax"][tmax_var].sel(warming_level=level),
        tmin=gwl_data["tmin"][tmin_var].sel(warming_level=level),
    )

summary_df = compute_report_metrics(
    periods,
    hot_day_threshold_F=hot_day_threshold_F,
    heatwave_min_days=heatwave_min_days,
    return_period_years=10,
)
display(summary_df)


## 6 · Derive headline stat-card numbers

Three numbers that surface the most-cited changes:

In [ ]:
hist_col = summary_df.columns[0]   # Historic baseline
proj_col = summary_df.columns[-1]  # Late-century (last warming level)

avg_high_row = next(r for r in summary_df.index if "Average High" in r)
hw_row       = next(r for r in summary_df.index if "Heat Wave Duration" in r)
hot_row      = next(r for r in summary_df.index if "Hot Days" in r)

delta_high = summary_df.loc[avg_high_row, proj_col] - summary_df.loc[avg_high_row, hist_col]
hist_hw    = summary_df.loc[hw_row, hist_col]
proj_hw    = summary_df.loc[hw_row, proj_col]
extra_hot  = summary_df.loc[hot_row, proj_col] - summary_df.loc[hot_row, hist_col]

hw_delta = proj_hw - hist_hw
hw_sign  = "+" if hw_delta >= 0 else ""

stat_items = [
    (f"+{delta_high:.1f}°F",          "Avg summer high\nvs historical"),
    (f"{hw_sign}{hw_delta:.1f} days", "Avg heat wave\nlength change"),
    (f"+{extra_hot:.0f}",             f"More hot days >{hot_day_threshold_F}°F/yr\nvs historical"),
]

for val, caption in stat_items:
    print(f"  {val:>12}  {caption!r}")


## 7 · Build bars-chart input

Shows the 1-in-10-yr extreme heat threshold across all four warming-level
periods so the chart matches the summary table above.


In [ ]:
extreme_row = next(r for r in summary_df.index if "1-in-" in r)

# Include all four warming-level periods so the bar chart matches the table
bars_df = pd.DataFrame(
    {col: [summary_df.loc[extreme_row, col]] for col in summary_df.columns},
    index=[county],
)
display(bars_df)


## 8 · Render the composite report figure

In [ ]:
fig = build_report_figure(
    title=f"{_short} County — Extreme Heat Climate Report",
    subtitle=(
        "Based on UCLA WRF daily climate model projections; "
        "scenarios represent global warming levels of 1.5°C, 2.0°C, "
        "and\n3.0°C above pre-industrial baseline."
    ),
    tagline=(
        f"{_short} County is already experiencing hotter, longer, and more "
        "frequent extreme heat events — conditions are expected to intensify "
        "in the decades ahead."
    ),
    stat_items=stat_items,
    summary_df=summary_df,
    bars_df=bars_df,
    bars_title="1-in-10-yr Daily Max Temperature (°F) by Warming Level",
    footnote=(
        "Y-axis truncated to highlight temperature differences; does not start at zero.\n"
        "Produced by Cal-Adapt: Analytics Engine's County Climate Report notebook."
    ),
)
display(fig)

## 9 · (Optional) Export to PNG / PDF

In [ ]:
# from pathlib import Path
#
# out_dir = Path("figures")
# out_dir.mkdir(exist_ok=True)
# slug = county.lower().replace(" ", "_")
#
# fig.savefig(out_dir / f"{slug}_climate_report.png", dpi=200, bbox_inches="tight")
# fig.savefig(out_dir / f"{slug}_climate_report.pdf",          bbox_inches="tight")
# print(f"Saved to {out_dir}/")